# Handoff note (Zoe → Sarah)

**What this notebook is:** baseline XGBRegressor for `viral_score` (0–100), temporal
70/15/15 split by `posted_at_local`, `n_estimators=300, max_depth=6, learning_rate=0.05`,
early stopping on the validation split.

**Model progression (same split & hyperparameters, fair comparison):**

| model | features | test R² |
|---|---|---|
| baseline | content features only (16 numeric) | ~0.06 |
| + categoricals | + `subreddit`, `category`, `engagement_mechanism` (19) | **~0.11** (nearly doubled) |

**SHAP takeaways (new model):**
- **`subreddit` is the #1 driver by a wide margin** (mean|SHAP| ≈ 2.2, ~1.7× the next feature),
  then `title_length`, then `engagement_mechanism`.
- **`category` is redundant with `subreddit`** — it ranked **last** (mean|SHAP| ≈ 0.0001) because
  the trees put all community signal into the finer-grained `subreddit`. Kept for interpretability
  only; drop it if you want a leaner model.

**Honest finding:** virality remains hard to predict — the model explains only **~11% of the
variance**. **`subreddit` (which community you post in) is the dominant signal**, while
content/emotion features (VADER, NRC emotions, readability) stay weak. Don't over-read the
content-feature effects.

**Data:** the feature table `zoe_nlp_features` now has **16 columns** (13 NLP/length + `subreddit`,
`category`, `engagement_mechanism`) and is published on HuggingFace
(`SPARC2026Reddit/MessyData-ZT`, `features/zoe_nlp_features.parquet`).

**Ownership:** Sarah owns final / segmented modeling from here (e.g. per-subreddit or
per-category models, richer text features, tuning).

---

# Baseline Model — XGBRegressor for viral_score

Predicts `viral_score` (0–100 virality index) from the engineered features in
`model_ready_dataset`.

- **Target (y):** `viral_score`
- **Features (X):** 19 columns — 16 numeric (temporal, media, length, VADER, 8 NRC
  emotions, readability) + 3 categorical (`subreddit`, `category`, `engagement_mechanism`)
  encoded as native XGBoost categoricals. `post_id` is dropped from X.
- **Split:** TEMPORAL (not random). Rows are ordered by `posted_at_local` and split
  70 / 15 / 15 → oldest 70% train, next 15% validation, newest 15% test. This mimics
  predicting the future from the past and avoids leakage.

## Phase 4A — setup + temporal split

In [ ]:
import duckdb
import numpy as np
import pandas as pd

# notebook lives in notebooks/zoe/, db is at project root
con = duckdb.connect("../../reddit_warehouse.db", read_only=True)

# Join posted_at_local from processed_posts so every row carries its timestamp for the
# temporal split. INNER JOIN on post_id (keys are 1:1 across tables).
df = con.execute("""
    SELECT m.*, p.posted_at_local
    FROM model_ready_dataset m
    JOIN processed_posts p ON m.post_id = p.post_id
""").df()
con.close()

df["posted_at_local"] = pd.to_datetime(df["posted_at_local"])
print("loaded:", df.shape)
print("timestamp nulls:", int(df["posted_at_local"].isna().sum()))
print("overall date range:", df["posted_at_local"].min(), "->", df["posted_at_local"].max())
df.head()

In [ ]:
# ── Separate X / y ────────────────────────────────────────────────────────
TARGET = "viral_score"
NUM_FEATURES = [
    "hour_of_day", "day_of_week", "has_media", "post_length_proxy",
    "vader_compound",
    "nrc_joy", "nrc_trust", "nrc_fear", "nrc_surprise",
    "nrc_sadness", "nrc_disgust", "nrc_anger", "nrc_anticipation",
    "readability", "title_length", "body_length",
]
# Categorical features. Encoded as pandas `category` dtype + XGBoost native categorical
# (enable_categorical=True) rather than one-hot: keeps each as ONE feature so SHAP ranking
# shows where subreddit/category land, and avoids ~60 one-hot columns (subreddit has 48 levels).
CAT_FEATURES = ["subreddit", "category", "engagement_mechanism"]
FEATURES = NUM_FEATURES + CAT_FEATURES

# ── TEMPORAL split (NOT random) ───────────────────────────────────────────
# Sort oldest -> newest; tie-break on post_id for determinism.
df_sorted = df.sort_values(["posted_at_local", "post_id"]).reset_index(drop=True)

# Fix category levels on the FULL dataset before slicing so train/val/test share the
# same category<->code mapping (otherwise XGBoost codes would misalign across splits).
for c in CAT_FEATURES:
    df_sorted[c] = df_sorted[c].astype("category")

n = len(df_sorted)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_sorted.iloc[:train_end]
val_df = df_sorted.iloc[train_end:val_end]
test_df = df_sorted.iloc[val_end:]

def make_Xy(part):
    X = part[FEATURES].copy()
    X["has_media"] = X["has_media"].astype(int)  # bool -> 0/1
    # CAT_FEATURES keep their (shared) category dtype from df_sorted
    y = part[TARGET].astype(float)
    return X, y

X_train, y_train = make_Xy(train_df)
X_val, y_val = make_Xy(val_df)
X_test, y_test = make_Xy(test_df)

print("=== TEMPORAL split (70/15/15 by posted_at_local) ===\n")
for name, part, X in [("TRAIN", train_df, X_train), ("VAL", val_df, X_val), ("TEST", test_df, X_test)]:
    print(f"{name:5} | rows={len(part):>6} | X.shape={X.shape} | nulls={int(X.isna().sum().sum())} "
          f"| {part['posted_at_local'].min()}  ->  {part['posted_at_local'].max()}")

# Leakage guard: train must end no later than val start, val no later than test start
assert train_df["posted_at_local"].max() <= val_df["posted_at_local"].min()
assert val_df["posted_at_local"].max() <= test_df["posted_at_local"].min()
print("\nordering check passed: train <= val <= test (no temporal overlap)")
print("total rows:", len(train_df) + len(val_df) + len(test_df), "of", n)

## Phase 4B — train baseline XGBRegressor

Simple baseline: `n_estimators=300, max_depth=6, learning_rate=0.05`, early stopping on the
validation split. Report RMSE / MAE / R² on TEST and TRAIN (overfitting check).

In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=20,
    enable_categorical=True,   # use native categorical splits for subreddit/category/engagement_mechanism
)

# Early stopping watches the (temporally later) validation split
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print("training complete")
print("best_iteration:", model.best_iteration, "of", model.n_estimators, "requested")
print("best validation RMSE:", round(model.best_score, 4))

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def report(y_true, y_pred):
    return (
        float(np.sqrt(mean_squared_error(y_true, y_pred))),
        float(mean_absolute_error(y_true, y_pred)),
        float(r2_score(y_true, y_pred)),
    )

pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

tr_rmse, tr_mae, tr_r2 = report(y_train, pred_train)
te_rmse, te_mae, te_r2 = report(y_test, pred_test)

print("=== metrics (viral_score, 0-100) ===")
print(f"{'':6}{'RMSE':>10}{'MAE':>10}{'R^2':>10}")
print(f"{'TRAIN':6}{tr_rmse:>10.3f}{tr_mae:>10.3f}{tr_r2:>10.4f}")
print(f"{'TEST':6}{te_rmse:>10.3f}{te_mae:>10.3f}{te_r2:>10.4f}")
print(f"\nreference: test y mean={y_test.mean():.2f}, std={y_test.std():.2f}")
print(f"overfitting gap (train R^2 - test R^2): {tr_r2 - te_r2:.4f}")

# ── Comparison vs the old 16-feature baseline (numeric-only) ───────────────
BASELINE_TEST_R2 = 0.06   # prior baseline before subreddit/category/engagement_mechanism
delta = te_r2 - BASELINE_TEST_R2
print("\n=== vs old baseline (16 numeric features, test R^2 ~0.06) ===")
print(f"new test R^2 = {te_r2:.4f}  |  baseline test R^2 = {BASELINE_TEST_R2:.2f}  |  delta = {delta:+.4f}")
print("IMPROVED" if delta > 0.005 else ("~UNCHANGED" if abs(delta) <= 0.005 else "REGRESSED"))

In [ ]:
# 10 test rows: actual vs predicted
compare = pd.DataFrame({
    "post_id": test_df["post_id"].values[:10],
    "actual_viral_score": y_test.values[:10],
    "predicted_viral_score": pred_test[:10],
})
compare["error"] = compare["predicted_viral_score"] - compare["actual_viral_score"]
print(compare.round(3).to_string(index=False))

## Phase 4C — interpret (SHAP) + track (MLflow)

SHAP feature importance over the test set, MLflow logging of params/metrics, and a
plain-language read of the top drivers.

In [ ]:
import shap
import xgboost as xgb
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for headless save
import matplotlib.pyplot as plt

# Exact TreeSHAP via XGBoost's native pred_contribs. Used instead of shap.TreeExplainer
# because the model has NATIVE categorical features; the booster computes exact SHAP
# values for categorical splits directly. Last column is the base value (dropped).
booster = model.get_booster()
dtest = xgb.DMatrix(X_test, enable_categorical=True)
contribs = booster.predict(dtest, pred_contribs=True)
shap_values = contribs[:, :-1]   # drop the bias/base-value column

# Ranked mean |SHAP| per feature (most -> least important)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_rank = pd.Series(mean_abs_shap, index=FEATURES).sort_values(ascending=False)

print("=== mean |SHAP| per feature (most -> least important) ===")
print(shap_rank.round(4).to_string())
print("\ncategorical feature ranks:")
for c in CAT_FEATURES:
    print(f"   {c:22} #{list(shap_rank.index).index(c) + 1} of {len(FEATURES)}   (mean|SHAP|={shap_rank[c]:.4f})")

# Beeswarm needs a numeric matrix for the color gradient; encode categoricals as codes.
X_test_plot = X_test.copy()
for c in CAT_FEATURES:
    X_test_plot[c] = X_test_plot[c].cat.codes
shap.summary_plot(shap_values, X_test_plot, feature_names=FEATURES, show=False)
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=120, bbox_inches="tight")
plt.close()
print("\nsaved SHAP summary plot -> shap_summary.png")

In [ ]:
import mlflow

mlflow.set_experiment("reddit_viral_baseline")
with mlflow.start_run(run_name="xgb_with_categoricals_temporal") as run:
    mlflow.log_params({
        "model": "XGBRegressor",
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "early_stopping_rounds": 20,
        "best_iteration": int(model.best_iteration),
        "split": "temporal_70_15_15_by_posted_at_local",
        "n_features": len(FEATURES),
        "cat_features": ",".join(CAT_FEATURES),
        "cat_encoding": "xgboost_native_categorical",
    })
    mlflow.log_metrics({
        "train_rmse": tr_rmse, "train_mae": tr_mae, "train_r2": tr_r2,
        "test_rmse": te_rmse,  "test_mae": te_mae,  "test_r2": te_r2,
    })
    # feature list logged as a JSON artifact (avoids param length limits)
    mlflow.log_dict({"features": FEATURES}, "feature_list.json")
    run_id = run.info.run_id

print("MLflow run created")
print("run_id:", run_id)
print("experiment:", mlflow.get_experiment_by_name("reddit_viral_baseline").experiment_id)

In [ ]:
# Top 5 drivers: direction (sign of SHAP-vs-feature correlation) + controllability.
# Numeric features are poster-controllable (timing / framing / emotion / length / media);
# subreddit & category are community CONTEXT (user picks where to post, but can't change the
# community's baseline behavior), engagement_mechanism is framing the poster controls.
CONTROLLABLE = {
    "hour_of_day": "timing", "day_of_week": "timing", "has_media": "media",
    "post_length_proxy": "length", "title_length": "length", "body_length": "length",
    "vader_compound": "framing/emotion", "readability": "framing",
    "nrc_joy": "emotion", "nrc_trust": "emotion", "nrc_fear": "emotion",
    "nrc_surprise": "emotion", "nrc_sadness": "emotion", "nrc_disgust": "emotion",
    "nrc_anger": "emotion", "nrc_anticipation": "emotion",
    "engagement_mechanism": "framing",
}
NON_CONTROLLABLE = {"subreddit": "community context", "category": "community context"}

print("=== Top 5 features driving viral_score (by mean |SHAP|) ===\n")
for f in shap_rank.head(5).index:
    i = FEATURES.index(f)
    if f in CAT_FEATURES:
        # correlation is meaningless on category codes; direction is level-specific
        msg = f"{f} matters but direction is category-specific (see SHAP plot)"
    else:
        corr = np.corrcoef(X_test[f].values.astype(float), shap_values[:, i])[0, 1]
        direction = "higher" if corr >= 0 else "lower"
        msg = f"higher {f} -> {direction} predicted virality"
    if f in NON_CONTROLLABLE:
        ctrl = f"NOT controllable ({NON_CONTROLLABLE[f]})"
    else:
        ctrl = f"CONTROLLABLE ({CONTROLLABLE.get(f, 'other')})"
    print(f"- {f} (mean|SHAP|={shap_rank[f]:.3f}): {msg}  [{ctrl}]")